# Inference with SOBER

Extract 512-dimensional cell embeddings and reconstruct masked protein intensities with the pretrained epoch-10 student model. This example uses the complete UPMC-HNC training region included in the repository; it demonstrates inference rather than held-out performance.

From the repository root:
```bash
python -m pip install -r requirements-tutorials.txt
python scripts/download_checkpoint.py
python -m jupyterlab tutorials/inference.ipynb
```
The download includes both student and EMA teacher weights. The analyses below use the student, as in the manuscript.

In [1]:
from pathlib import Path
import copy
import random
import sys

import numpy as np
import pandas as pd
import torch

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "sober/config.py").is_file())
sys.path.insert(0, str(ROOT))

from sober.checkpoint import load_checkpoint
from sober.data_loader import SpatialBiomarkerDataset
from sober.spatial_sampler import SpatialSampler

device = "cuda:0" if torch.cuda.is_available() else "cpu"
torch.set_num_threads(2)
model, checkpoint = load_checkpoint(ROOT / "checkpoints/sober_epoch10.pth", device=device)
print(f"Epoch {checkpoint['epoch']} | {len(checkpoint['biomarker_vocab'])} markers | {device}")

Loaded 41 biomarkers from sober/assets/biomarker_sequences.csv
Loaded 1282 biomarker aliases
Building biomarker vocabulary using learnable method...
Processing 41 valid biomarkers (EMPTY ones filtered out)
Building learnable embeddings...


Epoch 10 | 41 markers | cpu


## Load a tissue region

The loader aligns cell IDs and normalizes raw intensities per region and marker. The checkpoint supplies marker aliases and vocabulary. Marker names must match that vocabulary before inference.

In [2]:
data_config = copy.deepcopy(model.config)
data_config.use_cell_types = False
dataset = SpatialBiomarkerDataset(ROOT / "examples/data", data_config)
region = next(r for r in dataset.regions if r["study_name"] == "s240")
unknown = set(region["biomarkers"]) - set(checkpoint["biomarker_vocab"])
assert not unknown, f"Markers missing from the checkpoint: {sorted(unknown)}"
print(region["region_name"], region["num_cells"], "cells")

Loaded cell-type version table: 22 studies from sober/assets/study_celltype_final.tsv
Loaded 3 regions; skipped 0 small regions
UPMC_c001_v001_r001_reg039 1670 cells


## Extract cell embeddings

Each input contains a center cell and up to 100 neighbors within 300 stored coordinate units. Coordinates are relative to the center. This internal export retains native coordinate units; no physical calibration is assumed here.

Unmasked inputs produce one 512-dimensional vector per center cell. We use 16 evenly spaced center indices for this small example.

In [3]:
center_indices = np.linspace(0, region["num_cells"] - 1, 16, dtype=int)
sampler = SpatialSampler(model.config)
neighborhoods = [sampler.sample_region_at(region, int(i)) for i in center_indices]
model.masking = False
with torch.inference_mode():
    embeddings = model(neighborhoods, recon=False)["center_encoded"].cpu().numpy()
cell_ids = np.asarray([str(region["cell_ids"][i]) for i in center_indices])
assert embeddings.shape == (16, 512) and np.isfinite(embeddings).all()
print("Embedding shape:", embeddings.shape)
pd.DataFrame(embeddings[:, :5], index=cell_ids, columns=[f"dim_{i}" for i in range(5)]).head()

Embedding shape: (16, 512)


,dim_0,dim_1,dim_2,dim_3,dim_4
191876,1.130355,-0.176286,0.357210,-0.982559,-0.025596
191987,1.216575,-0.045099,0.228070,-0.923668,-0.116499
192098,-0.137264,-0.578590,1.314426,0.259447,-0.824944
192209,-0.127074,-0.570572,1.299876,0.248977,-0.836095
192321,-0.085857,-0.587494,1.268833,0.321546,-0.839715


## Reconstruct masked markers

The checkpoint masks 80% of center markers and independently samples a 20–80% masked count for each neighbor. Only masked center entries are scored. Predictions and observations are in normalized intensity units, not raw acquisition units.

In [4]:
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
model.masking = True
with torch.inference_mode():
    reconstructed = model(neighborhoods, recon=True)
model.masking = False

rows = reconstructed["target_descs"]
reconstruction = pd.DataFrame({
    "cell_id": [str(region["cell_ids"][row[2]]) for row in rows],
    "marker": [row[3] for row in rows],
    "observed": reconstructed["recon_target"].cpu().numpy(),
    "predicted": reconstructed["recon_results"].cpu().numpy(),
})
assert len(reconstruction) > 0 and np.isfinite(reconstruction[["observed", "predicted"]]).all().all()
reconstruction.head()

,cell_id,marker,observed,predicted
0,191876,CD11c,4.686837,4.704974
1,191876,CD134,4.507560,3.545180
2,191876,CD14,2.014359,1.308932
3,191876,CD15,1.182025,0.988323
4,191876,CD16,1.962946,3.766253


## Save the outputs

The embedding file retains cell IDs and center indices. The reconstruction table contains one row per masked center-cell marker.

In [5]:
output = ROOT / "checkpoints/inference_demo"
output.mkdir(parents=True, exist_ok=True)
np.savez_compressed(output / "cell_embeddings.npz", cell_ids=cell_ids,
                    center_indices=center_indices, embeddings=embeddings)
reconstruction.to_csv(output / "reconstruction.csv", index=False)
print("Saved cell_embeddings.npz and reconstruction.csv")

Saved cell_embeddings.npz and reconstruction.csv
